# Applying the xTC approximation

The xTC approximation is actually quite simple: you put the second quantized Hamiltonian in the normal order wrt the Hartree-Fock state, and then you remove the three-body terms. To implement this generalized normal ordering in openfermion, all you need is to switch from particles to holes in the occupied orbitals (that is done by swapping creation and annihilation operators on these modes). Then you just do the standard normal ordering.

In [1]:
import sys
sys.path.insert(0, "..")
from tc import *

import numpy as np
import pyscf.tools.fcidump
import openfermion as of
import matplotlib.pyplot as plt
from tqdm import tqdm

In [2]:
# two_path = "../outputs/Be_vmc_gpcc_1e6/FCIDUMP_tc_Be_vmc_gpcc_1e6_1"
# three_path = "../outputs/Be_vmc_gpcc_1e6/TCDUMP_tc_Be_vmc_gpcc_1e6_1"

# thresh=1e-15

# h1, h2, core = create_one_and_two_body(two_path, thresh)
# h3 = create_three_body(three_path, thresh)
# n_mo = len(h1)
# lattice = SquareLattice([n_mo], bc='closed')

In [3]:
def get_xtc_hamiltonian(op: of.FermionOperator,
                        occupied_orbitals: list) -> of.FermionOperator:
    h_flip = particle_hole_transform(op, occupied_orbitals)
    h_normal = of.normal_ordered(h_flip)
    return one_two_body(h_normal)


def particle_hole_transform(op: of.FermionOperator,
                            occupied_orbitals: list) -> of.FermionOperator:
    """Apply a particle-hole transformation on a fermionic operator.
    For the indicated orbitals, flip creation operators to annihilation 
    operators and vice versa. This corresponds to treating a Slater determinant
    with the indicated orbitals filled as 'physical vacuum',
    over which removing a particle creates a 'hole'."""
    op_new = of.FermionOperator()

    for k, v in op.terms.items():
        new_key = flip_ops(k, occupied_orbitals)
        op_new.terms[new_key] = v
    return op_new


def flip_ops(term_index: tuple, occupied_orbitals: list):
    new_index = []
    for a in term_index:
        if a[0] in occupied_orbitals:
            new_index.append((a[0], (a[1] + 1) % 2))
        else:
            new_index.append(a)
    return tuple(new_index)


def one_two_body(op: of.FermionOperator):
    op_new = of.FermionOperator()
    for k, v in op.terms.items():
        if len(k) <= 4:
            op_new.terms[k] = v
    return op_new

In [4]:
import openfermionpyscf

In [5]:
# h = openfermionpyscf.generate_molecular_hamiltonian(
#     "C 0 0 0", basis="sto6g", multiplicity=1
# )

# A = of.get_sparse_operator(h).todense()

# w, v = np.linalg.eigh(A)

# h_fo = of.get_fermion_operator(h)

# h_fo_flip = particle_hole_transform(h_fo, [0, 1, 2, 3, 4, 6])
# h_fo_flip = of.normal_ordered(h_fo_flip)
# B =  of.get_sparse_operator(h_fo_flip).todense()
# w_2, v_2 = np.linalg.eigh(B)

# np.max(w - w_2)

# len(of.normal_ordered(h_fo).terms)

# len(h_fo_flip.terms)

# h_tc = real_tc_hamilton_chemistry(lattice, h1, h2, h3,
#                                   threshold_terms=1e-15, spin_order="interleaved")

# h_tc_q = of.jordan_wigner(h_tc)

# len(h_tc_q.terms)

# h_tc_flip = particle_hole_transform(h_tc, [0, 1, 2, 3])

# h_tc_flip = of.normal_ordered(h_tc_flip)



# # h_xtc = one_two_body(h_tc_flip)
# h_xtc = get_xtc_hamiltonian(h_tc, [0, 1, 2, 3])

# h_xtc_qubit = of.jordan_wigner(h_xtc)
# print(len(h_xtc_qubit.terms))

# w, v = np.linalg.eig(of.get_sparse_operator(h_tc_q).todense())

# w_xtc, v_xtc = np.linalg.eig(of.get_sparse_operator(h_xtc_qubit).todense())

# print(np.sort(w.real)[:10])

# print(np.sort(w_xtc.real)[:10])

# plt.scatter(w_xtc.real, w_xtc.imag)

#### Build xTC Hamiltonians for folders

In [6]:
from pathlib import Path

In [7]:
import re

def natural_keys(text):
    """
    Key function for natural sorting. Splits string into text and number parts,
    converting number parts to integers.
    """
    
    # Pattern splits by digits (\d+) and keeps them as separate elements in the result
    return [int(c) if c.isdigit() else c for c in re.split(r'(\d+)', str(text))]

In [8]:
foldername="../outputs/water_different_bonds//"
# back to back spin order!
# occupied_orbitals = [0, 1, 2, 3, 4, 5, 6, 10, 11, 12, 13, 14, 15, 16]
occupied_orbitals = [0, 1, 2, 3, 4, 7, 8, 9, 10, 11]

p = Path(foldername)
g_fermi = sorted(list(p.glob("*sq_tc*")), key=natural_keys)

In [9]:
h = of.load_operator(g_fermi[0].name, 
                     data_directory=str(p),
                     plain_text=True)

In [10]:
g_fermi[0].name

'sq_tc_water_922_0.9.data'

In [11]:
%%time
for i, path in enumerate(g_fermi):
    print(i, path.name)
    h = of.load_operator(path.name, 
                     data_directory=str(p),
                     plain_text=True)    
    h_xtc_ph = get_xtc_hamiltonian(h, occupied_orbitals)
    h_xtc_qubit_ph = of.jordan_wigner(h_xtc_ph)

    h_xtc_p = particle_hole_transform(h_xtc_ph, occupied_orbitals)
    h_xtc_qubit_p = of.jordan_wigner(h_xtc_p)
    
    of.save_operator(h_xtc_ph, "h_xtc_ph_{0:}".format(i+1), data_directory=str(p),
                     plain_text=True, allow_overwrite=True)
    of.save_operator(h_xtc_qubit_ph, "h_xtc_qubit_ph_{0:}".format(i+1), 
                     data_directory=str(p),
                     plain_text=True, allow_overwrite=True)
    of.save_operator(h_xtc_p, "h_xtc_p_{0:}".format(i+1), data_directory=str(p),
                     plain_text=True, allow_overwrite=True)
    of.save_operator(h_xtc_qubit_p, "h_xtc_qubit_p_{0:}".format(i+1), 
                     data_directory=str(p),
                     plain_text=True, allow_overwrite=True)
    

0 sq_tc_water_922_0.9.data
1 sq_tc_water_922_1.0.data
2 sq_tc_water_922_1.1.data
3 sq_tc_water_922_1.2.data
4 sq_tc_water_922_1.3.data
5 sq_tc_water_922_1.4.data
6 sq_tc_water_922_1.5.data
7 sq_tc_water_922_1.6.data
8 sq_tc_water_922_1.7.data
9 sq_tc_water_922_1.8.data
10 sq_tc_water_922_1.9.data
11 sq_tc_water_922_2.0.data
12 sq_tc_water_922_2.1.data
13 sq_tc_water_922_2.2.data
CPU times: user 1min 2s, sys: 806 ms, total: 1min 3s
Wall time: 1min 3s


In [19]:
h_xtc_qubit_ph

(-2.827110605861395+0j) [] +
(-0.023543523894748876+0j) [X0 X1] +
(0.0012789361846825254+0j) [X0 X1 X2 Z3 Z4 X5] +
0.0015378640187916483j [X0 X1 X2 Z3 Z4 Y5] +
0.0004347904846499583j [X0 X1 Y2 Z3 Z4 X5] +
(0.001341874767616007+0j) [X0 X1 Y2 Z3 Z4 Y5] +
(-0.004180639172578687+0j) [X0 X1 Z2] +
(-0.0032177044115463023+0j) [X0 X1 Z3] +
(-0.003217704411546304+0j) [X0 X1 Z4] +
(-0.0015223321371429968+0j) [X0 X1 Z5] +
(0.010242672302532296+0j) [X0 X1 X6 X7] +
0.0020660719267505466j [X0 X1 X6 Y7] +
(0.0020167605904322307+0j) [X0 X1 X6 Z7 X8] +
-0.00039030894439564086j [X0 X1 X6 Z7 Y8] +
(0.01826525038212326+0j) [X0 X1 X6 Z7 Z8 Z9 Z10 X11] +
-0.004620812741782833j [X0 X1 X6 Z7 Z8 Z9 Z10 Y11] +
-0.0020660719267505466j [X0 X1 Y6 X7] +
(0.010242672302532296+0j) [X0 X1 Y6 Y7] +
-0.00039030894439564086j [X0 X1 Y6 Z7 X8] +
(-0.0020167605904322307+0j) [X0 X1 Y6 Z7 Y8] +
-0.004620812741782833j [X0 X1 Y6 Z7 Z8 Z9 Z10 X11] +
(-0.01826525038212326+0j) [X0 X1 Y6 Z7 Z8 Z9 Z10 Y11] +
(0.04307140368974141+0j)

In [20]:
h_xtc_qubit_p

(-2.827110605861395+0j) [] +
(0.023543523894748876+0j) [X0 X1] +
(-0.0012789361846825254+0j) [X0 X1 X2 Z3 Z4 X5] +
-0.0015378640187916483j [X0 X1 X2 Z3 Z4 Y5] +
-0.0004347904846499583j [X0 X1 Y2 Z3 Z4 X5] +
(-0.001341874767616007+0j) [X0 X1 Y2 Z3 Z4 Y5] +
(0.004180639172578687+0j) [X0 X1 Z2] +
(0.0032177044115463023+0j) [X0 X1 Z3] +
(0.003217704411546304+0j) [X0 X1 Z4] +
(0.0015223321371429968+0j) [X0 X1 Z5] +
(0.010242672302532296+0j) [X0 X1 X6 X7] +
-0.0020660719267505466j [X0 X1 X6 Y7] +
(0.0020167605904322307+0j) [X0 X1 X6 Z7 X8] +
-0.00039030894439564086j [X0 X1 X6 Z7 Y8] +
(0.01826525038212326+0j) [X0 X1 X6 Z7 Z8 Z9 Z10 X11] +
-0.004620812741782833j [X0 X1 X6 Z7 Z8 Z9 Z10 Y11] +
0.0020660719267505466j [X0 X1 Y6 X7] +
(0.010242672302532296+0j) [X0 X1 Y6 Y7] +
0.00039030894439564086j [X0 X1 Y6 Z7 X8] +
(0.0020167605904322307+0j) [X0 X1 Y6 Z7 Y8] +
0.004620812741782833j [X0 X1 Y6 Z7 Z8 Z9 Z10 X11] +
(0.01826525038212326+0j) [X0 X1 Y6 Z7 Z8 Z9 Z10 Y11] +
(0.04307140368974141+0j) [X0 

In [25]:
w, _ = np.linalg.eig(of.get_sparse_operator(h_xtc_qubit).todense())

In [26]:
np.min(w.real)

np.float64(-128.39228230972256)

In [27]:
w_, _ = np.linalg.eig(of.get_sparse_operator(h).todense())

In [28]:
np.min(w_.real)

np.float64(-128.3922823097202)